In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score

Audio-Guided Baseline (AGB)

In [ ]:


def bin_value_binary(x):

    if x <= thres:
        return "low"
    else:
        return "high"

def results_voice_music(clip):

  music_row = music_data[music_data["movie_clip"] == clip]
  voice_ratio=true_data[true_data["movie_clip"]== clip]["voice_ratio"].values[0]
  #print(f"clip name:{clip},voice ratio:{voice_ratio}")


  if not music_row.empty and music_row["Music Probability"].values[0] < 0.5 and voice_ratio<0.10:
      if clip in true_data["movie_clip"].values:
        data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","valence","true_val","val_bin_pred","audio_val","voice_ratio","Music Probability"]]

        #data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","arousal","true_aro","aro_actual","audio_aro","voice_ratio","Music Probability"]]

        # audio modality is useful and return audio expressed emotion
        return data_row["val_bin_pred"].values[0]

        #return data_row["audio_aro"].values[0]


  # uncomment this portion for arousal
  # elif not music_row.empty and music_row["Music Probability"].values[0] >=0.5:
  #     if clip in true_data["movie_clip"].values:
  #       #data_row=match_data[match_data["movie_clip"]==clip][["movie_clip","valence","true_val","val_bin_pred","audio_val","voice_ratio","Music Probability"]]
  #       data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","voice_ratio","Music Probability"]]
  #       #result_audio.append(data_row)
  #       return data_row["audio_aro"].values[0]

  elif(voice_ratio<=0.10):
      if clip in true_data["movie_clip"].values:
        data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","valence","true_val","val_actual","video_val","voice_ratio","Music Probability"]]

        #data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","arousal","true_aro","aro_actual","video_aro","voice_ratio","Music Probability"]]

        # video modality is useful and return video expressed emotion
        return data_row["val_actual"].values[0]
        #return data_row["video_aro"].values[0]



  elif(clip in true_data["movie_clip"].values):
      data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","valence","true_val","val_bin_pred","audio_val","voice_ratio","Music Probability"]]
      # return data_row["audio_val"].values[0]
      #data_row=true_data[true_data["movie_clip"]==clip][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","voice_ratio","Music Probability"]]

      # audio modality is useful and return audio expressed emotion
      return data_row["val_bin_pred"].values[0]
      #return data_row["audio_aro"].values[0]


def ccc(x, y):
    x_mean, y_mean = np.mean(x), np.mean(y)
    s_xy = np.mean((x - x_mean) * (y - y_mean))
    s_xx = np.var(x)
    s_yy = np.var(y)
    return (2 * s_xy) / (s_xx + s_yy + (x_mean - y_mean)**2)


In [ ]:

#read the attribute file
data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/genre_emotion.csv")
true_df=data.loc[:,["movie_clip","valence","arousal"]]
scale_column=["valence","arousal"]
scaler = MinMaxScaler(feature_range=(0, 1))
true_df[scale_column] = scaler.fit_transform(true_df[scale_column])
true_df[scale_column] = true_df[scale_column].round(2)
# files related to music and voice ratio
true_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/true_music_voice_valence_final.csv")
music_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/just_to_veriy_audio_video_valence_music.csv")
# for threshold only
movie_thres_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/movie_bin2_valence_info.csv")
movie_df=movie_thres_data.loc[:,["movie_name","mid_edge"]]

result_list=[]
result_merge=[]
audio_video_data=[]
for movie_name in os.listdir("/content/drive/MyDrive/PhdWork/NewProject(Movie)/audio_only"):
        pattern=movie_name
        print(f"printing for {pattern}")
        thres=movie_df[movie_df["movie_name"]==pattern]["mid_edge"].round(2).values[0]

        #read the video_subtile bin
        vid_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/subtitle_video_bin_pred2.csv")
        vid_data["movie_clip"]=vid_data["movie_clip"].str.split("/").str[1]
        match=vid_data["movie_clip"].str.contains(pattern)
        df1=vid_data.loc[match,["movie_clip","val_actual","aro_actual"]]
        df1[["val_actual","aro_actual"]]=df1[["val_actual","aro_actual"]].round(2)
        df1["video_val"]=df1["val_actual"].apply(bin_value_binary)

        #read the audio predicted file: audio data
        audio_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/audio_bin_pred2.csv")
        audio_data["movie_clip"]=audio_data["movie_clip"].str.split(".").str[0]
        audio_df=audio_data.loc[:,["movie_clip","val_bin_pred","aro_bin_pred"]]
        match=audio_df["movie_clip"].str.contains(pattern)
        df2=audio_df.loc[match,["movie_clip","val_bin_pred","aro_bin_pred"]]
        df2["audio_val"] = df2["val_bin_pred"].apply(bin_value_binary)

        match=true_df["movie_clip"].str.contains(pattern)
        df3=true_df.loc[match,["movie_clip","valence","arousal"]]
        df3["true_val"] = df3["valence"].apply(bin_value_binary)

        df_merged = df1.merge(df2, on="movie_clip", how="inner")
        df_merged = df_merged.merge(df3, on="movie_clip", how="inner")

        #print(df_merged.head(20))
        result_merge.append(df_merged)

final_result=pd.concat(result_merge,axis=0)
#print(final_result.head(20))
print(final_result.columns)
print(f"total samples:{final_result.shape}")
final_result = final_result.reset_index(drop=True)

filt_rows=final_result[(final_result["video_val"]!=final_result["true_val"]) & (final_result["audio_val"]!=final_result["true_val"])]

print(f"sample we are dropping:{filt_rows.shape}")
final_result=final_result.drop(filt_rows.index)
print(f"total samples after dropping:{final_result.shape}")

#print(final_result)
# If I directly use audio predicted output
# print(f"accuracy of only video predicted output: {accuracy_score(final_result['true_val'], final_result['video_val'])}")
# print(f"accuracy of only audio predicted output: {accuracy_score(final_result['true_val'], final_result['audio_val'])}")
# print(f"video:- precision: {precision_score(final_result['true_val'], final_result['video_val'],average="macro")}, recall:{recall_score(final_result['true_val'], final_result['video_val'],average="macro")},f1-score:{f1_score(final_result['true_val'], final_result['video_val'],average="macro")} ")
# print(f"audio:- precision: {precision_score(final_result['true_val'], final_result['audio_val'],average="macro")}, recall:{recall_score(final_result['true_val'], final_result['audio_val'],average="macro")},f1-score:{f1_score(final_result['true_val'], final_result['audio_val'],average="macro")} ")
#######################

# when audio and video predicted emotion is same
match_output=final_result[final_result["video_aro"]==final_result["audio_aro"]][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","aro_actual","video_aro"]]
match_output["pred_aro"] = match_output["video_aro"]  # since both are same

total_cases = len(final_result)
num_match_cases = len(match_output)
print(f"Matched cases: {num_match_cases} out of {total_cases} ({num_match_cases/total_cases:.2%})")

print(f"accuracy of match cases: {accuracy_score(match_output['true_aro'], match_output['video_aro'])}")
print(match_output[["arousal","aro_bin_pred","aro_actual"]])
print(ccc(match_output["arousal"],match_output["aro_actual"]))
disagree_output=final_result[final_result["video_aro"]!=final_result["audio_aro"]][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","aro_actual","video_aro"]]
#applying our method in diagreement cases
disagree_output["pred_aro"]=disagree_output["movie_clip"].apply(results_voice_music)
# blindly agree with audio_val in disagreement case
#disagree_output["pred_aro"]=disagree_output["audio_aro"]
print(f"accuracy of unmatch cases: {accuracy_score(disagree_output['true_aro'], disagree_output['pred_aro'])}")
#print(disagree_output)
#print(ccc(disagree_output["valence"],disagree_output["pred_val"]))
#print(match_output)


disagree_output = disagree_output.copy()
match_output = match_output.copy()

# Select consistent columns
match_output = match_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]
disagree_output = disagree_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]


final_pred_df = pd.concat([match_output, disagree_output], axis=0).reset_index(drop=True)


overall_acc = accuracy_score(final_pred_df["true_aro"], final_pred_df["pred_aro"])

overall_ccc = ccc(final_pred_df["arousal"], final_pred_df["aro_actual"])

print("Overall accuracy:", overall_acc)
print("Overall CCC:", overall_ccc)
print("Final merged dataframe shape:", final_pred_df.shape)



printing for TheFaultInOurStars
printing for Wonder
printing for LadyBird
printing for AboutTime
printing for TheSpectacularNow
printing for TheTheoryOfEverything
printing for TheJudge
printing for Gifted
printing for TheBlindSide
Index(['movie_clip', 'val_actual', 'aro_actual', 'video_val', 'val_bin_pred',
       'aro_bin_pred', 'audio_val', 'valence', 'arousal', 'true_val'],
      dtype='object')
total samples:(941, 10)
sample we are dropping:(102, 10)
total samples after dropping:(839, 10)
accuracy of only video predicted output: 0.6519666269368296
accuracy of only audio predicted output: 0.6746126340882003
video:- precision: 0.3259833134684148, recall:0.5,f1-score:0.39466089466089466 
audio:- precision: 0.7584070796460177, recall:0.7504570383912248,f1-score:0.6744627961745147 


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


'\nmatch_output=final_result[final_result["video_aro"]==final_result["audio_aro"]][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","aro_actual","video_aro"]]\nmatch_output["pred_aro"] = match_output["video_aro"]  # since both are same\n\ntotal_cases = len(final_result)\nnum_match_cases = len(match_output)\nprint(f"Matched cases: {num_match_cases} out of {total_cases} ({num_match_cases/total_cases:.2%})")\n\nprint(f"accuracy of match cases: {accuracy_score(match_output[\'true_aro\'], match_output[\'video_aro\'])}")\nprint(match_output[["arousal","aro_bin_pred","aro_actual"]])\nprint(ccc(match_output["arousal"],match_output["aro_actual"]))\ndisagree_output=final_result[final_result["video_aro"]!=final_result["audio_aro"]][["movie_clip","arousal","true_aro","aro_bin_pred","audio_aro","aro_actual","video_aro"]]\n#applying our method in diagreement cases\ndisagree_output["pred_aro"]=disagree_output["movie_clip"].apply(results_voice_music)\n# blindly agree with audio_val in dis

Video-Guided Baseline (VGB)

In [ ]:

def bin_value_binary(x):

    if x <= thres:
        return "low"
    else:
        return "high"


def results_voice_music(clip):

  '''
  This method decide if audio is useful

  return 1 means useful
  else non-useful
  '''

  music_row = music_data[music_data["movie_clip"] == clip]
  voice_ratio_series = true_data[true_data["movie_clip"]== clip]["voice_ratio"]
  voice_ratio = voice_ratio_series.values[0] if not voice_ratio_series.empty else -1.0 # Use -1.0 for safety

  music_prob = music_row["Music Probability"].values[0] if not music_row.empty else -1.0 # Use -1.0 for safety

  if music_prob != -1.0 and music_prob > 0.5 and voice_ratio != -1.0 and voice_ratio<0.10:
      return 1
  elif music_prob != -1.0 and music_prob >=0.5:
      return 0
  elif voice_ratio != -1.0 and voice_ratio<=0.10:
      return 0

  # Fallback to ensure a return value (default to audio useful)
  return 1


def ccc(x, y):
    x_mean, y_mean = np.mean(x), np.mean(y)
    s_xy = np.mean((x - x_mean) * (y - y_mean))
    s_xx = np.var(x)
    s_yy = np.var(y)
    return (2 * s_xy) / (s_xx + s_yy + (x_mean - y_mean)**2)

def results_video(clip):

  '''
  This method decide if video is useful

  return 1 means useful
  else non-useful
  '''

  data_row=video_status_data[video_status_data["video_name"].str.split(".").str[0]==clip][["video_name","usefulness_tool"]]
  if not data_row.empty:
    if data_row["usefulness_tool"].values[0]=="Useful":
      return 1
    else:
      return 0
  else:
    # Default to not useful (0) if the clip is not found in video_status_data
    return 0

def final_pred(clip):

    data_row=disagree_output[disagree_output["movie_clip"]==clip][["movie_clip","arousal","true_aro","audio_aro","pred_audio","video_aro","pred_video"]]
    #data_row=disagree_output[disagree_output["movie_clip"]==clip][["movie_clip","valence","true_val","audio_val","pred_audio","video_val","pred_video"]]
    if not data_row.empty:
      pred_audio_val = data_row["pred_audio"].values[0]
      pred_video_val = data_row["pred_video"].values[0]
      audio_aro_val = data_row["audio_aro"].values[0]
      video_aro_val = data_row["video_aro"].values[0]
      true_aro = data_row["true_aro"].values[0]

      # uncomment this when predicting based on usefullness of audio or video
      # change it as per audio or video whatever you going to use
      # if pred_video_val==1:
      #   return video_aro_val
      # else:
      #    return audio_aro_val


      # when both are useful or not useful, I checked accurcay when consider video_val or audio_val seperatly in those cases for final decision.

      if pred_audio_val==0 and pred_video_val==1:
        return video_aro_val
      elif pred_audio_val==1 and pred_video_val==0:
        return audio_aro_val
      # elif pred_audio_val==1 and pred_video_val==1:
      #   # for policy 1
      elif audio_aro_val==video_aro_val:
          return video_aro_val
      #   # for policy 2

      # elif audio_aro_val==true_aro:
      #     return audio_aro_val
      # elif video_aro_val==true_aro:
      #     return video_aro_val

        # Fallback if somehow they don't match (should be handled by drops)
        # else:
        #     return audio_aro_val # Default to audio if disagreement was not filtered

      # elif pred_audio_val==0 and pred_video_val==0:
      #   # This case should have been dropped by previous filtering
      #   return video_aro_val # Fallback, return one of them.

    # Fallback return: If data_row is empty or no conditions matched
    # This ensures final_pred always returns a string, preventing None issues.
    #return "low" # Arbitrary default to prevent None

    else:
      return np.nan


def final_pred_modality_stats(disagree_output):
    # Ensure all necessary columns are present
    required_cols = ["movie_clip", "true_aro", "audio_aro", "pred_audio", "video_aro", "pred_video"]

    for col in required_cols:
        if col not in disagree_output.columns:
            raise ValueError(f"Missing column: {col}")

    # (1) when pred_audio == 1 and pred_video == 1
    both_1 = disagree_output[(disagree_output["pred_audio"] == 1) & (disagree_output["pred_video"] == 1)]
    #print(both_1.head(50))
    both_1_match = (both_1["audio_aro"] == both_1["video_aro"]).sum()

    both_1_unmatch = len(both_1) - both_1_match
    both_1_audio_match_true = (((both_1["audio_aro"] == both_1["video_aro"])&(both_1["true_aro"] == both_1["audio_aro"])).sum())/both_1_match if both_1_match > 0 else float('nan')
    both_1_video_match_true = (((both_1["audio_aro"] == both_1["video_aro"])&(both_1["true_aro"] == both_1["video_aro"])).sum())/both_1_match if both_1_match > 0 else float('nan')

    both_1_audio_unmatch_true = (((both_1["audio_aro"] != both_1["video_aro"])&(both_1["true_aro"] == both_1["audio_aro"]).fillna(False)).sum())/both_1_unmatch if both_1_unmatch > 0 else float('nan')
    both_1_video_unmatch_true = (((both_1["audio_aro"] != both_1["video_aro"])&(both_1["true_aro"] == both_1["video_aro"]).fillna(False)).sum())/both_1_unmatch if both_1_unmatch > 0 else float('nan')

    both_1_audio_match_true_overall=(both_1["true_aro"] == both_1["audio_aro"]).sum()
    both_1_video_match_true_overall=(both_1["true_aro"] == both_1["video_aro"]).sum()
    # cross-mismatch details
    both_1_audio0_video1 = ((both_1["audio_aro"] == 'low') & (both_1["video_aro"] == 'high')).sum()
    both_1_audio1_video0 = ((both_1["audio_aro"] == 'high') & (both_1["video_aro"] == 'low')).sum()

    # (2) when pred_audio == 0 and pred_video == 0
    both_0 = disagree_output[(disagree_output["pred_audio"] == 0) & (disagree_output["pred_video"] == 0)]
    both_0_match = (both_0["audio_aro"] == both_0["video_aro"]).sum()
    both_0_unmatch = len(both_0) - both_0_match
    both_0_audio_match_true = (((both_0["audio_aro"] == both_0["video_aro"])&(both_0["true_aro"] == both_0["audio_aro"])).sum())/both_0_match if both_0_match > 0 else float('nan')
    both_0_video_match_true = (((both_0["audio_aro"] == both_0["video_aro"])&(both_0["true_aro"] == both_0["video_aro"])).sum())/both_0_match if both_0_match > 0 else float('nan')

    both_0_audio_match_video = (both_0["audio_aro"] == both_0["video_aro"]).sum()
    both_0_audio_unmatch_video = (both_0["audio_aro"] != both_0["video_aro"]).sum()

    both_0_audio_only_match_true = (both_0["audio_aro"] == both_0["true_aro"]).sum()
    both_0_video_only_match_true = (both_0["video_aro"] == both_0["true_aro"]).sum()

    both_0_audio_unmatch_true = (((both_0["audio_aro"] != both_0["video_aro"])&(both_0["true_aro"] == both_0["audio_aro"]).fillna(False)).sum())/both_0_unmatch if both_0_unmatch > 0 else float('nan')
    both_0_video_unmatch_true = (((both_0["audio_aro"] != both_0["video_aro"])&(both_0["true_aro"] == both_0["video_aro"]).fillna(False)).sum())/both_0_unmatch if both_0_unmatch > 0 else float('nan')

    # (3) when pred_audio == 0 and pred_video == 1
    a0v1 = disagree_output[(disagree_output["pred_audio"] == 0) & (disagree_output["pred_video"] == 1)]
    print(f"when audio is 0 and video is 1:{  len(a0v1)}")

    a0v1_audio_match_video = (a0v1["audio_aro"] == a0v1["video_aro"]).sum()
    a0v1_audio_unmatch_video = (a0v1["audio_aro"] != a0v1["video_aro"]).sum()

    a0v1_audio_match_true = (a0v1["true_aro"] == a0v1["audio_aro"]).sum()
    a0v1_video_match_true = (a0v1["true_aro"] == a0v1["video_aro"]).sum()

    # (4) when pred_audio == 1 and pred_video == 0
    a1v0 = disagree_output[(disagree_output["pred_audio"] == 1) & (disagree_output["pred_video"] == 0)]
    print(f"when audio is 1 and video is 0:{  len(a1v0)}")

    a1v0_audio_match_video = (a1v0["audio_aro"] == a1v0["video_aro"]).sum()
    a1v0_audio_unmatch_video = (a1v0["audio_aro"] != a1v0["video_aro"]).sum()

    a1v0_audio_match_true = (a1v0["true_aro"] == a1v0["audio_aro"]).sum()
    a1v0_video_match_true = (a1v0["true_aro"] == a1v0["video_aro"]).sum()

    # Print results
    print("=== (1) pred_audio=1, pred_video=1 ===")
    print(f"  Matches between audio_aro & video_aro: {both_1_match}")
    print(f"  Unmatches between audio_aro & video_aro: {both_1_unmatch}\n")
    print(f"  true_aro matches with audio_aro when Matches between audio_aro & video_aro : {both_1_audio_match_true}")
    print(f"  true_aro matches with video_aro when Matches between audio_aro & video_aro: {both_1_video_match_true}\n")
    print(f"  true_aro matches with audio_aro when Unmatches between audio_aro & video_aro : {both_1_audio_unmatch_true}")
    print(f"  true_aro matches with video_aro when Unmatches between audio_aro & video_aro: {both_1_video_unmatch_true}\n")

    print(f"  true_aro matches with audio_aro  : {both_1_audio_match_true_overall}")
    print(f"  true_aro matches with video_aro  : {both_1_video_match_true_overall}")



    print("=== (2) pred_audio=0, pred_video=0 ===")
    print(f"  Matches between audio_aro & video_aro: {both_0_match}")
    print(f"  Unmatches between audio_aro & video_aro: {both_0_unmatch}\n")


    print(f"  true_aro matches with audio_aro when Matches between audio_aro & video_aro: {both_0_audio_match_true}")
    print(f"  true_aro matches with video_aro when Matches between audio_aro & video_aro: {both_0_video_match_true}\n")
    print(f"  true_aro matches with audio_aro when Unmatches between audio_aro & video_aro: {both_0_audio_unmatch_true}")
    print(f"  true_aro matches with video_aro when Unmatches between audio_aro & video_aro: {both_0_video_unmatch_true}\n")

    print(f"  true_aro matches with audio_aro  : {both_0_audio_only_match_true}")
    print(f"  true_aro matches with video_aro  : {both_0_video_only_match_true}")

    print("=== (3) pred_audio=0, pred_video=1 ===")
    print(f"  Matches between audio_aro & video_aro: {a0v1_audio_match_video}")
    print(f"  Unmatches between audio_aro & video_aro: {a0v1_audio_unmatch_video}")

    print(f"  true_aro matches with audio_aro: {a0v1_audio_match_true}")
    print(f"  true_aro matches with video_aro: {a0v1_video_match_true}\n")

    print("=== (4) pred_audio=1, pred_video=0 ===")
    print(f"  Matches between audio_aro & video_aro: {a1v0_audio_match_video}")
    print(f"  Unmatches between audio_aro & video_aro: {a1v0_audio_unmatch_video}")

    print(f"  true_aro matches with audio_aro: {a1v0_audio_match_true}")
    print(f"  true_aro matches with video_aro: {a1v0_video_match_true}\n")

    # Optionally return dictionary for later use
    # return {
    #     "(1)_both_1": {"match": both_1_match, "unmatch": both_1_unmatch},
    #     "(2)_both_0": {"match": both_0_match, "unmatch": both_0_unmatch},
    #     "(3)_a0v1": {"true_audio_match": a0v1_audio_match_true, "true_video_match": a0v1_video_match_true},
    #     "(4)_a1v0": {"true_audio_match": a1v0_audio_match_true, "true_video_match": a1v0_video_match_true},
    # }

#read the attribute file
data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/genre_emotion.csv")
true_df=data.loc[:,["movie_clip","valence","arousal"]]
scale_column=["valence","arousal"]
scaler = MinMaxScaler(feature_range=(0, 1))
true_df[scale_column] = scaler.fit_transform(true_df[scale_column])
true_df[scale_column] = true_df[scale_column].round(2)
# files related to music and voice ratio
true_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/true_music_voice_arousal_final.csv")
music_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/just_to_veriy_audio_video_arousal_music.csv")
# files related to video useful or not
video_status_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/Emotion_Analysis_Results_with_matches_video.csv")
# for threshold only
movie_thres_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/movie_bin2_valence_info.csv")
movie_df=movie_thres_data.loc[:,["movie_name","mid_edge"]]

result_list=[]
result_merge=[]
audio_video_data=[]
movie_lst=[]
for movie_name in os.listdir("/content/drive/MyDrive/PhdWork/NewProject(Movie)/audio_only"):
        pattern=movie_name
        print(f"printing for {pattern}")
        thres=movie_df[movie_df["movie_name"]==pattern]["mid_edge"].round(2).values[0]
        movie_lst.append(pattern)
        #thres=0.8
        #read the video_subtile bin
        vid_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/subtitle_video_bin_pred2.csv")
        vid_data["movie_clip"]=vid_data["movie_clip"].str.split("/").str[1]
        match=vid_data["movie_clip"].str.contains(pattern)
        df1=vid_data.loc[match,["movie_clip","val_actual","aro_actual"]]
        df1[["val_actual","aro_actual"]]=df1[["val_actual","aro_actual"]].round(2)
        df1["video_aro"]=df1["aro_actual"].apply(bin_value_binary)

        #read the audio predicted file: audio data
        audio_data=pd.read_csv("/content/drive/MyDrive/PhdWork/NewProject(Movie)/input_files/audio_bin_pred2.csv")
        audio_data["movie_clip"]=audio_data["movie_clip"].str.split(".").str[0]
        audio_df=audio_data.loc[:,["movie_clip","val_bin_pred","aro_bin_pred"]]
        match=audio_df["movie_clip"].str.contains(pattern)
        df2=audio_df.loc[match,["movie_clip","val_bin_pred","aro_bin_pred"]]
        df2["audio_aro"] = df2["aro_bin_pred"].apply(bin_value_binary)

        match=true_df["movie_clip"].str.contains(pattern)
        df3=true_df.loc[match,["movie_clip","valence","arousal"]]
        df3["true_aro"] = df3["arousal"].apply(bin_value_binary)

        df_merged = df1.merge(df2, on="movie_clip", how="inner")
        df_merged = df_merged.merge(df3, on="movie_clip", how="inner")

        #print(df_merged.head(20))
        result_merge.append(df_merged)

final_result=pd.concat(result_merge,axis=0)
final_result["movie_name"]=final_result['movie_clip'].str.rsplit('_', n=1).str[0]
clip_counts = final_result['movie_name'].value_counts()
print(clip_counts)

#print(final_result.head(20))
print(final_result.columns)
final_result = final_result.reset_index(drop=True)
print(f"number of clips before filtering:{final_result.shape[0]}")
filt_rows=final_result[(final_result["video_aro"]!=final_result["true_aro"]) & (final_result["audio_aro"]!=final_result["true_aro"])]
filt_rows["pred_audio"]=filt_rows["movie_clip"].apply(results_voice_music)
filt_rows["pred_video"]=filt_rows["movie_clip"].apply(results_video)
clip_counts_discarded = filt_rows['movie_name'].value_counts()
print(clip_counts_discarded)
distribution = (
    filt_rows
    .groupby(['movie_name', 'pred_audio', 'pred_video'])
    .size()
    .unstack(['pred_audio', 'pred_video'], fill_value=0)
)

print(distribution)
distribution_pct = distribution.div(distribution.sum(axis=1), axis=0)

avg_percentage = distribution_pct.mean(axis=0) * 100
print(avg_percentage)

#print(filt_rows.head(20))
final_result=final_result.drop(filt_rows.index)
print(f"after filtering the rows:{final_result.shape[0]}")

# working with final_result: pred_audio and pred_video indicate if modality is useful or non-useful
final_result["pred_audio"]=final_result["movie_clip"].apply(results_voice_music)
final_result["pred_video"]=final_result["movie_clip"].apply(results_video)
# call this function for statistics
#final_pred_modality_stats(final_result)

#######
'''
match_output=final_result[final_result["video_val"]==final_result["audio_val"]][["movie_clip","valence","true_val","val_bin_pred","audio_val","val_actual","video_val"]]
match_output["pred_val"] = match_output["video_val"]  # since both are same

total_cases = len(final_result)
num_match_cases = len(match_output)
print(f"Matched cases: {num_match_cases} out of {total_cases} ({num_match_cases/total_cases:.2%})")

print(f"accuracy of match cases: {accuracy_score(match_output['true_val'], match_output['pred_val'])}")
#print(match_output[["arousal","aro_bin_pred","aro_actual"]])
print(ccc(match_output["valence"],match_output["val_actual"]))
'''

disagree_output=final_result.copy()# add this line to apply policy 1 or policy 2
'''
disagree_output=final_result[final_result["video_val"]!=final_result["audio_val"]][["movie_clip","valence","true_val","val_bin_pred","audio_val","val_actual","video_val"]]
print(f"length of disagree_output:{disagree_output.shape[0]}")
print(f"cases where video val matches with true_val:{disagree_output[disagree_output['video_val']==disagree_output['true_val']].shape[0]}")
print(f"cases where audio val matches with true_val:{disagree_output[disagree_output['audio_val']==disagree_output['true_val']].shape[0]}")
'''
#applying our method in diagreement cases
# checking if audio is useful
#disagree_output["pred_aro"]=disagree_output["movie_clip"].apply(results_voice_music)
# checking if video is useful
#disagree_output["pred_val"]=disagree_output["movie_clip"].apply(results_video)

# disagree_output["pred_audio"]=disagree_output["movie_clip"].apply(results_voice_music)
# disagree_output["pred_video"]=disagree_output["movie_clip"].apply(results_video)
# combining output from both tool audio and video
print(f"length of disagree_output:{disagree_output.shape[0]}")
#print(disagree_output[(disagree_output["pred_audio"]==0) & (disagree_output["pred_video"]==0) ].head(50))
disagree_output.drop(disagree_output[(disagree_output["pred_audio"]==0) & (disagree_output["pred_video"]==0) ].index,inplace=True)
print(f"after filtering the rows both are non useful:{disagree_output.shape[0]}")
# uncomment below line when apply for policy 1
disagree_output.drop(
    disagree_output[
        ((disagree_output["pred_audio"] == 1) & (disagree_output["pred_video"] == 1))
        & (disagree_output["audio_aro"] != disagree_output["video_aro"])
    ].index,
    inplace=True
)
print(f"after filtering the rows where audio and video is useful, their expressed emotion is not matching:{disagree_output.shape[0]}")
disagree_output["pred_aro"]=disagree_output["movie_clip"].apply(final_pred)

#print(disagree_output["pred_aro"].isna().sum())
assert disagree_output["pred_aro"].notna().all(), \
    "pred_aro contains NaN values!"

# use final_pred_modality_stats method to analyse the results
final_pred_modality_stats(disagree_output)

# blindly agree with audio_val in disagreement case
#disagree_output["pred_aro"]=disagree_output["audio_aro"]
print(f"accuracy of unmatch cases: {accuracy_score(disagree_output['true_aro'], disagree_output['pred_aro'])}")
print(f"precision: {precision_score(disagree_output['true_aro'], disagree_output['pred_aro'],average="macro")}, recall:{recall_score(disagree_output['true_aro'], disagree_output['pred_aro'],average="macro")},f1-score:{f1_score(disagree_output['true_aro'], disagree_output['pred_aro'],average="macro")} ")
print(f"data points in disagree:{disagree_output.shape[0]}")
#print(ccc(disagree_output["valence"],disagree_output["pred_val"]))
#print(match_output)

'''
disagree_output = disagree_output.copy()
match_output = match_output.copy()

# Select consistent columns
match_output = match_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]
disagree_output = disagree_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]


final_pred_df = pd.concat([match_output, disagree_output], axis=0).reset_index(drop=True)


overall_acc = accuracy_score(final_pred_df["true_aro"], final_pred_df["pred_aro"])
overall_ccc = ccc(final_pred_df["arousal"], final_pred_df["aro_actual"])

print("Overall accuracy:", overall_acc)
print("Overall CCC:", overall_ccc)
print("Final merged dataframe shape:", final_pred_df.shape)
'''


printing for TheFaultInOurStars
printing for Wonder
printing for LadyBird
printing for AboutTime
printing for TheSpectacularNow
printing for TheTheoryOfEverything
printing for TheJudge
printing for Gifted
printing for TheBlindSide
movie_name
Wonder                   155
TheJudge                 154
AboutTime                137
LadyBird                 114
TheBlindSide              94
TheSpectacularNow         83
TheFaultInOurStars        78
Gifted                    67
TheTheoryOfEverything     59
Name: count, dtype: int64
Index(['movie_clip', 'val_actual', 'aro_actual', 'video_aro', 'val_bin_pred',
       'aro_bin_pred', 'audio_aro', 'valence', 'arousal', 'true_aro',
       'movie_name'],
      dtype='object')
number of clips before filtering:941


/tmp/ipykernel_10550/4235711333.py:278: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_rows["pred_audio"]=filt_rows["movie_clip"].apply(results_voice_music)
/tmp/ipykernel_10550/4235711333.py:279: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt_rows["pred_video"]=filt_rows["movie_clip"].apply(results_video)


movie_name
Wonder                   34
LadyBird                 28
TheBlindSide             24
AboutTime                16
TheJudge                 12
TheFaultInOurStars       11
TheSpectacularNow         7
TheTheoryOfEverything     6
Gifted                    4
Name: count, dtype: int64
pred_audio              1     0   
pred_video              0  1  0  1
movie_name                        
AboutTime               9  7  0  0
Gifted                  3  0  1  0
LadyBird               24  4  0  0
TheBlindSide           20  4  0  0
TheFaultInOurStars      6  4  1  0
TheJudge                8  3  1  0
TheSpectacularNow       6  0  1  0
TheTheoryOfEverything   1  3  1  1
Wonder                 26  7  0  1
pred_audio  pred_video
1           0             66.706809
            1             22.961584
0           0              8.152958
            1              2.178649
dtype: float64
after filtering the rows:799
length of disagree_output:799
after filtering the rows both are non useful:754
a

'\ndisagree_output = disagree_output.copy()\nmatch_output = match_output.copy()\n\n# Select consistent columns\nmatch_output = match_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]\ndisagree_output = disagree_output[["movie_clip", "arousal", "true_aro", "pred_aro", "audio_aro", "aro_actual", "video_aro"]]\n\n\nfinal_pred_df = pd.concat([match_output, disagree_output], axis=0).reset_index(drop=True)\n\n\noverall_acc = accuracy_score(final_pred_df["true_aro"], final_pred_df["pred_aro"])\noverall_ccc = ccc(final_pred_df["arousal"], final_pred_df["aro_actual"])\n\nprint("Overall accuracy:", overall_acc)\nprint("Overall CCC:", overall_ccc)\nprint("Final merged dataframe shape:", final_pred_df.shape)\n'